## Setting the Database Connection

Connect to Postgres using the `get_ibis_connection` function:

In [1]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from sql_ai_agent.data import get_ibis_connection

Set a connection to the Postgres database:

In [3]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

Extract the table attributes using the `get_tbl_attr` function:

In [ ]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name="air_traffic")
schema = tbl_attr.schema

Extract distinct values from the categorical fields:

In [5]:
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt

distinct_values = get_character_distinct_values(
    con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
)
distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)

print(distinct_values)
print(distinct_values_formatted)

{'Operating Airline': ['ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China', 'Air Europe', 'Air France', 'Air India Limited', 'Air Italy S.P.A', 'Air New Zealand', 'Air Pacific Limited dba Fiji Airways', 'Air Premia, Inc.', 'AirTran Airways', 'Air Transat', 'Air Wisconsin', 'Alaska Airlines', 'Alitalia Airlines', 'Allegiant Air', 'Allegro Airlines', 'All Nippon Company Airways, Ltd.', 'American Airlines', 'American Eagle Airlines', 'Ameriflight', 'Asiana Airlines', 'ATA Airlines', 'Atlantic Southeast Airlines', 'Atlas Air, Inc', 'BelAir Airlines', 'Boeing Company', 'Breeze Aviation Group, Inc.', 'British Airways', 'Canadian Airlines', 'Casino Express', 'Cathay Pacific', 'Champion Air', 'China Airlines', 'China Eastern', 'China Eastern Airlines, Inc', 'China Southern', 'Comair', 'Compass Airlines', 'Condor Flugdienst GmbH', '

## Setting Up Prompt Templates
- Use the `ChatPromptTemplate` function to set the prompt template
- Add Additional Context: add a new argument named as `additional_context`.
- Add Memory: add message place holder using the `MessagesPlaceholder` method.

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()
system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"
messages = [
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", user_template),
]
prompt_template = ChatPromptTemplate.from_messages(messages)

## Setting the LLM Client

Use the `ChatOpenAI` to set the LLM client connection:

In [ ]:
from langchain_openai import ChatOpenAI

base_url = "https://api.deepseek.com/v1"
model = "deepseek-chat"

api_key=""

llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    temperature=0,
    model=model
)

Create the chain:

In [8]:
chain = prompt_template | llm

In [20]:
def basic_sql_agent(chain, question, tbl_name, schema, con, chat_history, additional_context=""):
    
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
            "chat_history": chat_history.messages,
        }
    )
    query = llm_output.content

    chat_history.add_user_message(question)
    chat_history.add_ai_message(query)
    print("The return SQL query:")
    print(query)
    output = con.sql(query).execute()
    print(output)
    return output, chat_history


In [21]:
output, chat_history = basic_sql_agent(
    chain,
    question="How many passengers departed during 2024 via terminal 1?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    chat_history=chat_history,
    con=con,
)

The return SQL query:
SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2024 AND "Activity Type Code" = 'Deplaned' AND "Terminal" = 'Terminal 1';
       sum
0  6991489


In [22]:
output, chat_history = basic_sql_agent(
    chain,
    question="And via terminal 2?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    chat_history=chat_history,
    con=con,
)

The return SQL query:
SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2024 AND "Activity Type Code" = 'Deplaned' AND "Terminal" = 'Terminal 2';
       sum
0  2974867


In [12]:
print(chat_history)

Human: How many passengers departed during 2024 via terminal 1?
AI: SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2024 AND "Activity Type Code" = 'Deplaned' AND "Terminal" = 'Terminal 1';
Human: And via terminal 2?
AI: SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2024 AND "Activity Type Code" = 'Deplaned' AND "Terminal" = 'Terminal 2';


## Using Skills to Extend the Agent Capabilities
Extend the agent’s context by providing additional information—such as business logic, data mappings, or domain-specific rules—that cannot be reliably inferred from the data alone.

## Loading and Injecting Skills

In [23]:
from sql_ai_agent.skill_manager import SkillManager

# Initialize skill manager
skill_manager = SkillManager(skills_dir= "../skills")

# Load the SFO air traffic skill
sfo_skill = skill_manager.load_skill("sfo_air_traffic_context")

print(sfo_skill)

# SFO Air Traffic Passenger Statistics - Domain Knowledge

## Schema Reference

```
Table: air_traffic (Monthly passenger data, July 1999 - present)

Year                        int64      -- 4-digit year (1999-present)
Date                        timestamp  -- First day of month (YYYY-MM-DD)
Operating Airline           string     -- Airline that operated the flight
Operating Airline IATA Code string     -- 2-letter code (may be NULL)
Published Airline           string     -- Marketing airline (codeshare)
Published Airline IATA Code string     -- 2-letter code (may be NULL)
GEO Summary                 string     -- "Domestic" or "International"
GEO Region                  string     -- "US", "Europe", "Asia", etc.
Activity Type Code          string     -- "Deplaned", "Enplaned", "Thru / Transit"
Price Category Code         string     -- "Low Fare" or "Other"
Terminal                    string     -- "Terminal 1", "Terminal 2", etc.
Boarding Area               string     -- A, B, C, D, 

In [ ]:
from sql_ai_agent.SqlAgent import SqlAgent
agent_with_skill = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    tbl_name=tbl_name,
    fallback=False,
    fallback_model=model,
    skill=True,
    skills_dir="../skills",
)

In [27]:
complex_question = """
  Which international airlines showed the biggest growth from 2023 to 2024?
  Show the top 10 airlines ranked by percentage increase in passenger volume.
  For each airline, show their 2023 total, 2024 total, and calculate year-over-year growth

  Make sure to exclude transit passengers and avoid double-counting from code share agreements.
  """

result_with_skill = agent_with_skill.ask_question(
    question=complex_question,
    distinct_char_values = True,
    verbose=True,
    trials= 0
)

ℹ️  Query modified to enforce LIMIT: 10000
Error in the query processing, trying to debug...

❌ QUERY FAILED

Query:
WITH totals AS (
  SELECT
    "Operating Airline" AS airline,
    SUM(CASE WHEN "Year" = 2023 THEN "Passenger Count" ELSE 0 END) AS total_2023,
    SUM(CASE WHEN "Year" = 2024 THEN "Passenger Count" ELSE 0 END) AS total_2024
  FROM air_traffic
  WHERE
    "Year" IN (2023, 2024)
    AND "GEO Summary" = 'International'
    AND "Activity Type Code" IN ('Deplaned', 'Enplaned')
  GROUP BY
    "Operating Airline"
)
SELECT
  airline,
  total_2023,
  total_2024,
  (
    (
      total_2024 - total_2023
    ) * 100.0
  ) / NULLIF(total_2023, 0) AS growth_percentage
FROM totals
WHERE
  total_2023 > 0
ORDER BY
  growth_percentage DESC
LIMIT 10

Error: WITH query name "totals" specified more than once
LINE 1: ...aned', 'Enplaned') GROUP BY "Operating Airline"), totals AS ...
                                                             ^
